# Experiment 23: Odds-Anchored Calibration (Recent International Matches)

First pass on candidate #2: blend market-implied 1X2 probabilities into Poisson score calibration.

Notes:
- Uses publicly available `WorldCup2026.csv` odds/results feed (covers 2023+ internationals).
- This is **not** a full LOTO WC backtest because historical 2010/2014/2018 market files are not available in this source.
- Goal: validate whether odds-anchored reweighting can improve recent out-of-time score predictions.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from evaluation.sporza import sporza_points_series

ODDS_URL = 'https://www.football-data.co.uk/WorldCup2026.csv'
MAX_GOALS = 8

BASE_FEATURES_HOME = [
    'home_elo', 'away_elo', 'elo_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
BASE_FEATURES_AWAY = [
    'away_elo', 'home_elo', 'elo_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]

In [ ]:
NAME_MAP = {
    'United States': 'USA',
    'Bosnia and Herzegovina': 'Bosnia & Herzegovina',
    'Czechia': 'Czech Republic',
    'Democratic Republic of Congo': 'DR Congo',
    'Korea, South': 'South Korea',
    "Cote d'Ivoire": 'Ivory Coast',
    'Bosnia-Herzegovina': 'Bosnia & Herzegovina',
    'Curacao': 'Curaçao',
}

def harmonize_team(name: str) -> str:
    return NAME_MAP.get(name, name)

def build_X(df: pd.DataFrame, features: list[str]) -> np.ndarray:
    x = df[features].copy()
    return x.fillna(x.median(numeric_only=True)).values

def build_pipe(alpha: float = 0.1) -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poisson', PoissonRegressor(alpha=alpha, max_iter=400)),
    ])

def implied_probs(h_odds: float, d_odds: float, a_odds: float) -> tuple[float, float, float]:
    inv = np.array([1.0 / h_odds, 1.0 / d_odds, 1.0 / a_odds], dtype=float)
    inv = np.where(np.isfinite(inv), inv, 0.0)
    s = inv.sum()
    if s <= 0:
        return (np.nan, np.nan, np.nan)
    p = inv / s
    return float(p[0]), float(p[1]), float(p[2])

def score_joint_probs(lh: float, la: float) -> np.ndarray:
    goals = np.arange(MAX_GOALS + 1)
    joint = np.outer(poisson.pmf(goals, lh), poisson.pmf(goals, la))
    return joint / joint.sum()

def outcome_probs_from_joint(joint: np.ndarray) -> tuple[float, float, float]:
    p_h = 0.0
    p_d = 0.0
    p_a = 0.0
    for h in range(joint.shape[0]):
        for a in range(joint.shape[1]):
            p = joint[h, a]
            if h > a:
                p_h += p
            elif h == a:
                p_d += p
            else:
                p_a += p
    return p_h, p_d, p_a

def reweight_joint_with_market(joint: np.ndarray, market_probs: tuple[float, float, float], blend_w: float, eps: float = 1e-8) -> np.ndarray:
    q_h, q_d, q_a = outcome_probs_from_joint(joint)
    m_h, m_d, m_a = market_probs

    r_h = (1.0 - blend_w) * q_h + blend_w * m_h
    r_d = (1.0 - blend_w) * q_d + blend_w * m_d
    r_a = (1.0 - blend_w) * q_a + blend_w * m_a

    w_h = r_h / max(q_h, eps)
    w_d = r_d / max(q_d, eps)
    w_a = r_a / max(q_a, eps)

    out = joint.copy()
    for h in range(out.shape[0]):
        for a in range(out.shape[1]):
            if h > a:
                out[h, a] *= w_h
            elif h == a:
                out[h, a] *= w_d
            else:
                out[h, a] *= w_a
    out = out / out.sum()
    return out

def best_score_for_sporza(joint: np.ndarray) -> tuple[int, int]:
    best = (-1.0, 1, 0)
    for ph in range(joint.shape[0]):
        for pa in range(joint.shape[1]):
            exp_pts = 0.0
            for ah in range(joint.shape[0]):
                for aa in range(joint.shape[1]):
                    p = joint[ah, aa]
                    if ph == ah and pa == aa:
                        exp_pts += p * 10
                    elif (ph - pa) == (ah - aa):
                        exp_pts += p * 7
                    elif np.sign(ph - pa) == np.sign(ah - aa):
                        exp_pts += p * 5
                    else:
                        exp_pts += p
            if exp_pts > best[0]:
                best = (exp_pts, ph, pa)
    return best[1], best[2]

In [ ]:
features = pd.read_parquet('../../data/interim/train_features.parquet').copy()
features['date'] = pd.to_datetime(features['date'])

odds_raw = pd.read_csv(ODDS_URL, encoding='utf-8-sig')
odds = odds_raw.rename(columns={'Home': 'home_team', 'Away': 'away_team'}).copy()
odds['date'] = pd.to_datetime(odds['Date'], dayfirst=True, errors='coerce')
odds['home_team'] = odds['home_team'].astype(str).map(harmonize_team)
odds['away_team'] = odds['away_team'].astype(str).map(harmonize_team)
for c in ['H_Avg', 'D_Avg', 'A_Avg']:
    odds[c] = pd.to_numeric(odds[c], errors='coerce')
odds[['p_h_mkt', 'p_d_mkt', 'p_a_mkt']] = odds.apply(
    lambda r: implied_probs(r['H_Avg'], r['D_Avg'], r['A_Avg']), axis=1, result_type='expand'
)

merge_cols = ['date', 'home_team', 'away_team', 'p_h_mkt', 'p_d_mkt', 'p_a_mkt']
df = features.merge(odds[merge_cols], on=['date', 'home_team', 'away_team'], how='inner')
df = df.dropna(subset=['home_score', 'away_score', 'p_h_mkt', 'p_d_mkt', 'p_a_mkt']).copy()

print(f'Feature rows total: {len(features):,}')
print(f'Rows with matched odds: {len(df):,} ({len(df)/len(features):.1%})')
print('Matched rows by year:')
print(df['date'].dt.year.value_counts().sort_index().to_string())

In [ ]:
# Temporal split (recent out-of-time backtest)
train = df[df['date'] < '2025-01-01'].copy()
test = df[df['date'] >= '2025-01-01'].copy()

# Keep a small validation tail inside train for blend tuning
val = train[train['date'] >= '2024-07-01'].copy()
fit = train[train['date'] < '2024-07-01'].copy()

print(f'fit={len(fit)}, val={len(val)}, test={len(test)}')

pipe = build_pipe(alpha=0.1)
X_fit = np.vstack([build_X(fit, BASE_FEATURES_HOME), build_X(fit, BASE_FEATURES_AWAY)])
y_fit = np.concatenate([fit['home_score'].values, fit['away_score'].values])
pipe.fit(X_fit, y_fit)

def eval_split(d: pd.DataFrame, blend_w: float):
    lh = np.clip(pipe.predict(build_X(d, BASE_FEATURES_HOME)), 0.01, None)
    la = np.clip(pipe.predict(build_X(d, BASE_FEATURES_AWAY)), 0.01, None)

    base_pred_h, base_pred_a, cal_pred_h, cal_pred_a = [], [], [], []
    for i, row in enumerate(d.itertuples(index=False)):
        joint = score_joint_probs(float(lh[i]), float(la[i]))
        b_h, b_a = best_score_for_sporza(joint)
        base_pred_h.append(b_h)
        base_pred_a.append(b_a)

        joint_cal = reweight_joint_with_market(joint, (row.p_h_mkt, row.p_d_mkt, row.p_a_mkt), blend_w)
        c_h, c_a = best_score_for_sporza(joint_cal)
        cal_pred_h.append(c_h)
        cal_pred_a.append(c_a)

    base_pts = sporza_points_series(pd.Series(base_pred_h), pd.Series(base_pred_a), d['home_score'].reset_index(drop=True), d['away_score'].reset_index(drop=True))
    cal_pts = sporza_points_series(pd.Series(cal_pred_h), pd.Series(cal_pred_a), d['home_score'].reset_index(drop=True), d['away_score'].reset_index(drop=True))
    return float(base_pts.mean()), float(cal_pts.mean())

grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
val_rows = []
for w in grid:
    b, c = eval_split(val, w)
    val_rows.append({'blend_w': w, 'base_val_pts': b, 'cal_val_pts': c, 'delta': c - b})
val_df = pd.DataFrame(val_rows).sort_values('cal_val_pts', ascending=False)
best_w = float(val_df.iloc[0]['blend_w'])
print('Validation tuning (sorted):')
print(val_df.to_string(index=False))
print(f'\nSelected blend_w={best_w:.2f}')

base_test, cal_test = eval_split(test, best_w)
print(f'\nTest mean Sporza pts — base: {base_test:.3f}, odds-calibrated: {cal_test:.3f}, delta: {cal_test - base_test:+.3f}')